In [23]:
import sys
sys.path.append('../../utils')
import os 
from functions import * 
from importlib import reload

# Path to the Leaflet repository
PATH_TO_LEAFLET_REPO = '/gpfs/commons/home/kisaev/Leaflet/src/beta-binomial-mix/'
sys.path.append(PATH_TO_LEAFLET_REPO)

import betabinomo_mix_singlecells
import load_cluster_data

#from load_cluster_data import load_cluster_data
from betabinomo_mix_singlecells import *

import torch
import sklearn.manifold 
import time
from tqdm import tqdm
import seaborn as sns
sns.set_theme(style="whitegrid")

### Settings and Load data

In [24]:
# Get all organ names in /gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters
organ_names = os.listdir('/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters')
organ_names = [organ_name for organ_name in organ_names if '.gz' not in organ_name]
organ_names = [organ_name for organ_name in organ_names if '.bed' not in organ_name]
organ_names = [organ_name for organ_name in organ_names if 'sparse_matrices' not in organ_name]
organ_names

['Trachea',
 'Eye',
 'Lung',
 'Liver',
 'Mammary',
 'Spleen',
 'Vasculature',
 'Salivary_Gland',
 'Kidney',
 'Thymus',
 'Pancreas',
 'Bone_Marrow',
 'Small_Intestine',
 'Heart',
 'Uterus',
 'Muscle',
 'Lymph_Node',
 'Tongue',
 'Prostate',
 'Blood',
 'Fat',
 'Large_Intestine']

In [25]:
def generate_matrices(organ):

    input_files_folder="/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/"+organ+"/"

    _, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = load_cluster_data.load_cluster_data(
        input_folder = input_files_folder, has_genes="yes") 

    print("The number of cells is: ", len(cell_ids_conversion))
    print("The number of junctions is: ", len(junction_ids_conversion))
    print("The number of intron clusters observed is: ", len(junction_ids_conversion.Cluster.unique()))
    print("The number of genes is: ", len(junction_ids_conversion.gene_id.unique()))

    # save coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion into folder output_file="/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/sparse_matrices/"+organ
    output_file="/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/sparse_matrices/"+organ+"/"
    # check if this folder exists
    if not os.path.exists(output_file):
        os.makedirs(output_file)
        # save coo_counts_sparse
    torch.save(coo_counts_sparse, output_file+"coo_counts_sparse.pt")
    # save coo_cluster_sparse
    torch.save(coo_cluster_sparse, output_file+"coo_cluster_sparse.pt")
    # save cell_ids_conversion
    torch.save(cell_ids_conversion, output_file+"cell_ids_conversion.pt")
    # save junction_ids_conversion
    torch.save(junction_ids_conversion, output_file+"junction_ids_conversion.pt")

    print("Done!")

In [32]:
# read in coo_cluster_sparse.pt
torch.load("/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/sparse_matrices/Liver/junction_ids_conversion.pt")

,junction_id_index,junction_id,Cluster,gene_id
115267,0,10_100235764_100236849,54035,ENSG00000095485.18
16710,1,10_100235764_100238021,54035,ENSG00000095485.18
115268,2,10_100236969_100238021,54035,ENSG00000095485.18
21623,3,10_100246935_100250247,54039,ENSG00000095485.18
81165,4,10_100246935_100253420,54039,ENSG00000095485.18
...,...,...,...,...
28817,25083,Y_284314_288732,104768,ENSG00000182378.15_PAR_Y
47440,25084,Y_284314_290647,104768,ENSG00000182378.15_PAR_Y
28818,25085,Y_288869_290647,104768,ENSG00000182378.15_PAR_Y
92204,25086,Y_3003217_3058656,104814,ENSG00000231535.8


In [26]:
for org in organ_names:
    print(org)
    generate_matrices(org)

Trachea
Reading in data from folder ...
/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/Trachea/
Finished reading in data from folder ...
['mucus secreting cell' 'B cell' 'secretory cell' 'goblet cell'
 'macrophage' 'tracheal goblet cell' 'neutrophil' 'smooth muscle cell'
 'fibroblast' 'endothelial cell' 'ionocyte'
 'serous cell of epithelium of trachea' 'plasma cell' 'T cell'
 'ciliated cell' 'mast cell' 'basal cell']
472
64512
                                             cell_id  Cluster  Cluster_Counts  \
0  TSP6_Trachea_NA_SS2_Blue_B133898_Epithelial_A1...      110              81   
1  TSP6_Trachea_NA_SS2_Blue_B133898_Epithelial_A1...      110              81   
2  TSP6_Trachea_NA_SS2_Blue_B133898_Epithelial_A1...      110              81   
3  TSP6_Trachea_NA_SS2_Blue_B133898_Epithelial_A1...      112              53   
4  TSP6_Trachea_NA_SS2_Blue_B133898_Epithelial_A1...      309               3   

             junction_id             gene_id 